# micro.py — 15-Min Bar Move Timing (Walk-Forward)

**Purpose:** For SPY 15-minute bars over the last 60 trading days, find bars
where volume and volatility are both elevated (a "signal"), then walk
forward in time from each signal to measure: how many bars it took for
price to move 2 standard deviations away from the signal price, what the
volume and volatility looked like at the moment of the signal, and what the
resulting percent price move was. This is walk-forward by construction —
every rolling stat only looks backward at each point in time, no future
data leaks into a signal's own threshold.

**Data:** SPY 15-minute OHLCV via yfinance, free — 60 days ending
2026-08-14 (Friday), which happens to match yfinance's free intraday
history limit exactly, so no paid data is needed for this pass.

**Design choices (flag if you want these changed):**
- "Volatility" = rolling standard deviation of 15-min log returns over the
  trailing 20 bars (~5 hours) — a short local vol estimate, recomputed at
  every bar using only past data (walk-forward, no lookahead).
- "2 std move" = 2x that bar's trailing volatility, in log-return terms —
  i.e. how big a move the recent past says is a genuine 2-sigma event,
  not a fixed dollar or percent number.
- Signal condition = volume AND volatility both above their own 60-day
  median at that point — a rough, adjustable threshold, not a tuned one.
- Output: for every signal, bars-to-move, volume-at-signal,
  volatility-at-signal, and % move achieved — so you can see the
  relationship between all four, not just whether it "worked."

**Total code: ~50 lines across 2 cells, per your spec.**


In [ ]:
# --- Block 1: fetch 15-min bars, compute rolling volume + volatility ---
import yfinance as yf
import pandas as pd
import numpy as np

# 60 days of 15-min SPY bars, ending Friday 2026-08-14 (yfinance's free cap)
spy = yf.download("SPY", period="60d", interval="15m", end="2026-08-15")
spy = spy[["Open", "High", "Low", "Close", "Volume"]].dropna()

# 15-min log returns
spy["Return"] = np.log(spy["Close"] / spy["Close"].shift(1))

# Rolling (backward-looking only) volume and volatility, ~5 hours = 20 bars
window = 20
spy["VolRoll"] = spy["Volume"].rolling(window).mean()
spy["VolatilityRoll"] = spy["Return"].rolling(window).std()

# Signal: both volume and volatility above their own median for this window
vol_thresh = spy["VolRoll"].median()
volt_thresh = spy["VolatilityRoll"].median()
spy["Signal"] = (spy["VolRoll"] > vol_thresh) & (spy["VolatilityRoll"] > volt_thresh)

print(f"Total 15-min bars: {len(spy)}")
print(f"Signal bars flagged: {spy['Signal'].sum()}")


In [ ]:
# --- Block 2: walk forward from each signal, measure bars-to-2std-move ---
results = []
max_lookahead = 40  # cap search at 40 bars (~10 hours) forward

for sig_time in spy.index[spy["Signal"]]:
    idx = spy.index.get_loc(sig_time)
    sigma = spy["VolatilityRoll"].iloc[idx]
    entry_price = spy["Close"].iloc[idx]
    target_move = 2 * sigma  # 2-std threshold, in log-return terms

    for j in range(idx + 1, min(idx + max_lookahead, len(spy))):
        cum_return = np.log(spy["Close"].iloc[j] / entry_price)
        if abs(cum_return) >= target_move:
            results.append({
                "bars_to_move": j - idx,
                "volume_at_signal": spy["VolRoll"].iloc[idx],
                "volatility_at_signal": sigma,
                "pct_move": cum_return * 100,
            })
            break

results_df = pd.DataFrame(results)
print(f"Signals that reached a 2-std move within {max_lookahead} bars: {len(results_df)}")
print(results_df.describe())
